## Telco Customer Churn Dataset ##
**Objective**: Predict customer retention based on the customer metadata, like services that each customer has signed up for, customer account and demographic information

## 1. Config and Imports

In [68]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [69]:
pd.set_option('display.max_columns', None)

In [70]:
DATA_PATH = "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

NUMERIC_COLUMNS = ['tenure', 'MonthlyCharges', 'TotalCharges']
BOOLEAN_COLUMNS = ['SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 
                   'PaperlessBilling']

## 2. Helper Functions

### 2.1 Read & Overview

In [71]:
def read_data(path: str) -> pd.DataFrame:
    """Reads the data from the given path and returns a pandas DataFrame."""
    return pd.read_csv(path)

In [72]:
def check_data_quality(df: pd.DataFrame) -> pd.DataFrame:
    """Checks the quality of the data and returns a summary table."""
    print(f"Number of rows: {df.shape[0]}")
    print(f"Number of columns: {df.shape[1]}")
    print(f"Duplicated rows: {df.duplicated().sum()}\n")
    
    # Create summary table
    summary = pd.DataFrame({
        'Column': df.columns,
        'Data Type': df.dtypes.values,
        'Missing Values': df.isna().sum().values,
        'Missing %': (df.isna().sum().values / len(df) * 100).round(2),
        'Unique Values': df.nunique().values
    })
    
    return summary

In [73]:
def convert_to_boolean(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Converts specified columns in the DataFrame to boolean type."""
    result = df.copy()

    values_mapping = {'Yes': True, 'No': False, 0 : False, 1: True}

    for col in columns:
        if result[col].isin(values_mapping.keys()).all():
            result[col] = result[col].map(values_mapping).astype(bool)
        else:
            raise ValueError(f"Column '{col}' contains values not in the mapping.")
    return result

In [74]:
def convert_to_numeric(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Converts specified columns in the DataFrame to numeric type."""

    result = df.copy()

    for col in columns:
        values = result[col].replace(r"^\s*$", np.nan, regex=True)
        result[col] = pd.to_numeric(values, errors="raise")
    return result

## 3. Read & Overview Data ##
Based on information provided in kaggle dataset description:
* **Unique values**: CustomerID
* **Boolean values**: SeniorCitizen, Partner, Dependents, PhoneService, PaperlessBilling

In [75]:
df = read_data(DATA_PATH)
df = convert_to_boolean(df, BOOLEAN_COLUMNS)
df = convert_to_numeric(df, NUMERIC_COLUMNS)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,False,True,False,1,False,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,True,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,False,False,False,34,True,No,DSL,Yes,No,Yes,No,No,No,One year,False,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,False,False,False,2,True,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,True,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,False,False,False,45,False,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,False,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,False,False,False,2,True,No,Fiber optic,No,No,No,No,No,No,Month-to-month,True,Electronic check,70.70,151.65,Yes


In [76]:
check_data_quality(df)

Number of rows: 7043
Number of columns: 21
Duplicated rows: 0



,Column,Data Type,Missing Values,Missing %,Unique Values
0,customerID,object,0,0.00,7043
1,gender,object,0,0.00,2
2,SeniorCitizen,bool,0,0.00,2
3,Partner,bool,0,0.00,2
4,Dependents,bool,0,0.00,2
5,tenure,int64,0,0.00,73
6,PhoneService,bool,0,0.00,2
7,MultipleLines,object,0,0.00,3
8,InternetService,object,0,0.00,3
9,OnlineSecurity,object,0,0.00,3


There are **11 missing values** in TotalCharges columns

In [77]:
df[df['TotalCharges'].isna()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,False,True,True,0,False,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,True,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,False,False,True,0,True,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,False,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,False,True,True,0,True,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,False,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,False,True,True,0,True,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,False,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,False,True,True,0,False,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,False,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,False,True,True,0,True,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,False,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,False,True,True,0,True,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,False,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,False,True,True,0,True,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,False,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,False,True,True,0,True,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,True,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,False,True,True,0,True,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,False,Mailed check,73.35,NaN,No


In all cases when TotalCharge is nan, the column **tenure** is equal 0. That might indicate a new customer, and in fact TotalCharge should be equal 0. 
* If there are no other cases in which tenure is equal zero, these customers will be treated as new, and TotalCharge will be filled with 0.
* Otherwise these values are missing, and these cases will be excluded

In [78]:
print(f"Number of rows with tenure = 0: {df[df['tenure'] == 0].shape[0]}")

Number of rows with tenure = 0: 11


In [79]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

#### Summary ####
* **TotalCharges** contains 11 missing values, originally represented by whitespaces. All affected customers have tenure (number of months customer was active) is equal 0, therefore these customers are treated as new, and the missing values have been filled with 0.